# Video inference smoke test (without the app)
This notebook runs a lightweight, end-to-end pass on a local video using the model weights available in the repo. It avoids the web app and focuses on file I/O, model loading, inference, and simple visualization.

**Reminder:** Activate your env first (e.g., `conda activate dl_env` or source your venv).

# 1) Set Up Environment and Dependencies
- Installs/imports core deps (torch, opencv-python, numpy, matplotlib).
- Sets key paths (video, weights, output).
- Selects device (CPU/GPU).

In [ ]:
# If needed, uncomment to install deps in this env (avoid doing this repeatedly)
# %pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
# %pip install opencv-python numpy matplotlib
# %pip install supervision inference-gpu transformers git+https://github.com/roboflow/sports.git

import os
import sys
import time
from pathlib import Path
from typing import Dict, Any

import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt

# Paths — adjust to your files
repo_root = Path("..").resolve()  # Atlas-RN-Sport
workspace_root = repo_root.parent  # football_tactical_analysis (contains video_processor.py and models/)
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))
if str(workspace_root) not in sys.path:
    sys.path.append(str(workspace_root))

video_path = repo_root / "video_data" / "0bfacc_0.mp4"
weights_path = workspace_root / "models" / "tactical_vulnerability_complete.pth"
output_path = repo_root / "video_data" / "sample_annotated.mp4"
output_path.parent.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
roboflow_api_key = os.getenv("ROBOFLOW_API_KEY", "").strip()
print(f"Using device: {device}")
print(f"Video: {video_path if video_path.exists() else 'MISSING'}")
print(f"Weights: {weights_path if weights_path.exists() else 'MISSING'}")
print(f"Roboflow API key set: {'yes' if roboflow_api_key else 'no'}")

: 

# 2) Load Pretrained Model/Weights
- Replace the placeholder model with your actual architecture.
- Ensure the checkpoint matches the model class before loading state_dict.

# Where to plug the real detector/EGNN
The notebook still uses a `DummyModel` that draws random boxes. Replace it with your actual detection/EGNN pipeline here.

# 2A) Full pipeline: YOLO + ByteTrack + team classifier + tactical heuristics
- Uses the real `FootballVideoProcessor` (Roboflow YOLO + ByteTrack + SigLIP team classifier) and the tactical analyzer in `backend/ai_service/app/real_processor.py`.
- Requires a Roboflow API key (`ROBOFLOW_API_KEY`) and these packages in the active env: `supervision`, `inference-gpu`, `transformers`, `sports` (Roboflow sports utils).
- Sets `stride`/`max_frames` for a faster smoke test; bump them for a full run. Outputs threat/weakness zones and vulnerability scores.

In [ ]:
# Real pipeline run (set ROBOFLOW_API_KEY first)
from backend.ai_service.app.real_processor import get_analyzer

if not roboflow_api_key:
    raise RuntimeError("Set ROBOFLOW_API_KEY in your env to run the full pipeline.")

analyzer = get_analyzer(roboflow_api_key=roboflow_api_key)
if not analyzer.video_processor:
    raise RuntimeError("Video processor failed to initialize (check Roboflow key and deps).")

def progress_callback(pct: float, msg: str):
    print(f"{pct:5.1f}% | {msg}")

# Run detection + tracking + team classification
tracking_data = analyzer.video_processor.process_video(
    video_path=str(video_path),
    confidence=0.3,
    stride=2,
    max_frames=180,
    progress_callback=progress_callback,
)

# Run tactical analysis on the resulting tracking (same logic as analyzer.analyze_video)
full_results = analyzer._analyze_tracking_data(tracking_data)

print("\n--- Summary ---")
print(full_results.get("summary", {}))
print("\nThreat zones (top 3):")
print(full_results.get("threat_zones", [])[:3])
print("\nWeakness zones (top 3):")
print(full_results.get("weakness_zones", [])[:3])
print("\nVulnerability scores:")
print(full_results.get("vulnerability_scores", {}))

In [ ]:
# Example scaffold for your real model
# TODO: replace with your architecture + preprocessing + postprocessing
# from your_model_lib import YourModelClass, build_transforms

# def load_real_model(weights: Path) -> torch.nn.Module:
#     model = YourModelClass()
#     ckpt = torch.load(weights, map_location=device)
#     state = ckpt.get("state_dict", ckpt)
#     model.load_state_dict(state, strict=True)
#     model.to(device).eval()
#     return model
#
# real_model = load_real_model(weights_path)
#
# def preprocess_frame(frame: np.ndarray) -> torch.Tensor:
#     # apply your transforms (resize/normalize/permute) matching training
#     tensor = build_transforms()(frame)  # placeholder
#     return tensor.to(device).unsqueeze(0)
#
# def decode_predictions(pred: Dict[str, Any]):
#     # convert model outputs to [N,4] boxes in normalized coords, scores, labels
#     boxes = pred["boxes"]  # torch tensor shape [N,4] xyxy normalized
#     scores = pred.get("scores", torch.ones(len(boxes), device=boxes.device))
#     labels = pred.get("labels", torch.zeros(len(boxes), device=boxes.device))
#     return boxes.detach().cpu().numpy(), scores.detach().cpu().numpy(), labels.detach().cpu().numpy()
#
# # In the main loop, swap DummyModel usage:
# #   pred = real_model(inp)
# #   boxes, scores, labels = decode_predictions(pred)
# #   frame_annot = draw_boxes(frame.copy(), boxes)


In [ ]:
class DummyModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("_noop", torch.tensor(0.0))

    def forward(self, x: torch.Tensor) -> Dict[str, Any]:
        # Return a fake prediction: one box center per frame
        b = x.shape[0]
        # centers and sizes in normalized coords
        centers = torch.rand(b, 2, device=x.device)
        sizes = torch.full((b, 2), 0.2, device=x.device)
        return {
            "boxes": torch.cat([centers - sizes / 2, centers + sizes / 2], dim=-1),  # [b,4]
            "scores": torch.rand(b, device=x.device),
            "labels": torch.zeros(b, dtype=torch.long, device=x.device),
        }


def load_model(weights: Path) -> torch.nn.Module:
    model = DummyModel().to(device).eval()
    if weights.exists():
        try:
            state = torch.load(weights, map_location=device)
            if isinstance(state, dict) and "state_dict" in state:
                state = state["state_dict"]
            model.load_state_dict(state, strict=False)
            print(f"Loaded weights from {weights}")
        except Exception as e:
            print(f"Warning: could not load weights ({e}); using dummy weights.")
    else:
        print("Weights not found, using dummy model.")
    return model


model = load_model(weights_path)

# 3) Configure Video Input Source
- Point to your local video file (set `video_path`).
- Inspect FPS, frame size, and codec to ensure OpenCV can read it.

In [ ]:
def open_video(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Video not found: {path}")
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video opened: {path} | {w}x{h} @ {fps:.2f} fps | frames: {count}")
    return cap, fps, (w, h), count


cap, fps, frame_size, frame_count = open_video(video_path)

# 4) Run Inference on Video Frames
- Preprocess frames (resize/normalize as your model expects).
- Run the model and collect predictions.
- This uses a dummy model; replace preprocessing + decoding with your pipeline.

In [ ]:
def preprocess_frame(frame: np.ndarray) -> torch.Tensor:
    # Example: resize to 224x224 and normalize to [0,1]
    resized = cv2.resize(frame, (224, 224))
    tensor = torch.from_numpy(resized).float() / 255.0
    tensor = tensor.permute(2, 0, 1).unsqueeze(0)  # [1,3,H,W]
    return tensor.to(device)


def draw_boxes(frame: np.ndarray, boxes: np.ndarray, color=(0, 255, 0)) -> np.ndarray:
    h, w, _ = frame.shape
    for x1, y1, x2, y2 in boxes:
        cv2.rectangle(frame, (int(x1 * w), int(y1 * h)), (int(x2 * w), int(y2 * h)), color, 2)
    return frame


max_frames = 200  # adjust for quick smoke test
writer = cv2.VideoWriter(
    str(output_path),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps if fps > 0 else 25,
    frame_size,
)

processed = 0
start = time.time()
sample_frame = None

while processed < max_frames:
    ok, frame = cap.read()
    if not ok:
        break

    # Preprocess and infer
    inp = preprocess_frame(frame)
    with torch.no_grad():
        pred = model(inp)

    boxes = pred.get("boxes", torch.zeros((1, 4), device=device)).detach().cpu().numpy()
    frame_annot = draw_boxes(frame.copy(), boxes)

    writer.write(frame_annot)
    processed += 1
    if sample_frame is None:
        sample_frame = frame_annot.copy()

end = time.time()
cap.release()
writer.release()

print(f"Processed {processed} frames in {end - start:.2f}s ({processed / max(end-start,1e-6):.2f} fps)")

# 5) Visualize and Save Output Video
- Shows one annotated frame inline.
- Output video is written to `output_path`.

In [ ]:
if sample_frame is None:
    raise RuntimeError("No frame processed. Check video path and codecs.")

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(sample_frame, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Sample annotated frame")
plt.show()

print(f"Annotated video saved to: {output_path}")

# 6) Batch Testing and Basic Metrics
- Optionally iterate multiple videos.
- Log throughput and frame counts.
- Customize success criteria (e.g., boxes above a score threshold).

In [ ]:
test_videos = [video_path]

metrics = []
for vid in test_videos:
    if not vid.exists():
        print(f"Skip missing: {vid}")
        continue
    # For brevity, reuse prior inference by inspecting output_path size
    size_mb = output_path.stat().st_size / 1e6 if output_path.exists() else 0
    metrics.append({
        "video": str(vid),
        "frames_processed": processed,
        "fps_effective": processed / max(end - start, 1e-6),
        "output_mb": round(size_mb, 2),
    })

print(metrics if metrics else "No metrics collected.")

# 7) Cleanup and Next Steps
- Release resources and outline what to plug in next (real model, preprocessing, decoding).
- Ideas: integrate EGNN pipeline, hook to real tracker/detector outputs, compare with app responses.

In [ ]:
# Cleanup (everything already released, but add hooks if you open windows)
cv2.destroyAllWindows()

print("Done. Next steps:")
print("- Swap DummyModel with your actual architecture + preprocessing.")
print("- Point weights_path to your real checkpoint.")
print("- Tune preprocess_frame and draw_boxes to match your model outputs.")
print("- If you have detector/tracker outputs, feed them into EGNN pipeline here.")